# Phân tích SQL - Dữ liệu thương mại điện tử Olist

Notebook này chạy 8 câu truy vấn Q1–Q8 từ tệp `sql/03_queries.sql` bằng SQLite và pandas.

### Mục tiêu phân tích
- Kiểm tra ảnh hưởng của ngày đặc biệt và biến động doanh số.
- Đánh giá phân bố và mức cân bằng của `review_score`.
- So sánh giao trễ, phí vận chuyển, độ phức tạp đơn hàng, thanh toán và khu vực với mức hài lòng.


## Tóm tắt và tổng hợp notebook

### Mục đích
Notebook phân tích dữ liệu Brazilian E-Commerce Public Dataset by Olist giai đoạn 2016–2018 nhằm trả lời hai câu hỏi trọng tâm:
- Những yếu tố nào trong quá trình mua và giao hàng chi phối mức độ hài lòng `review_score`?
- Có thể dự đoán review tại thời điểm đặt hàng hay chỉ sau khi giao hàng mà không vi phạm data leakage?

### Mục tiêu
Notebook kết nối SQLite với `data/olist.db`, tải view và đặc trưng từ `sql/03_queries.sql`, đồng thời tổng hợp item, thanh toán và người bán theo từng đơn hàng trước khi phân tích để tránh nhân đôi dữ liệu.

8 truy vấn được tổ chức thành các nhóm mục tiêu:
- **Q1:** Phân bố điểm đánh giá và mức mất cân bằng lớp.
- **Q2–Q3:** Mối quan hệ giữa giao trễ, mức độ trễ và điểm review.
- **Q4:** Tổng phí ship và tỷ lệ phí ship trên giá trị hàng hóa.
- **Q5:** Độ phức tạp qua số sản phẩm, số mặt hàng và số người bán trong đơn.
- **Q6:** Phương thức thanh toán và số kỳ trả góp.
- **Q7:** Xu hướng review theo thời gian 22 tháng.
- **Q8:** Phân hóa mức hài lòng theo bang và khu vực khách hàng.

### Đầu ra và cách sử dụng
- Mỗi truy vấn trả về một `DataFrame`, dùng cho so sánh, trực quan hóa và rút kết luận kinh doanh/logistics.
- `v_model_features_at_purchase` chỉ chứa thông tin có tại thời điểm đặt hàng và phù hợp cho mô hình dự đoán sớm.
- `v_model_features_post_delivery` bổ sung biến giao hàng và chỉ phù hợp khi dự đoán sau khi đơn hàng đã được giao.
- Việc tách hai tập feature giúp kiểm soát data leakage và hỗ trợ câu hỏi nghiên cứu thứ hai một cách minh bạch.

In [20]:
import sqlite3
from pathlib import Path

import pandas as pd

base_dir = Path.cwd().resolve()
if base_dir.name == "Notebooks":
    base_dir = base_dir.parent

db_path = base_dir / "data" / "olist.db"
sql_path = base_dir / "sql" / "03_queries.sql"

if not db_path.exists():
    raise FileNotFoundError(f"Khong tim thay database: {db_path}")

print("Database:", db_path)
print("SQL source:", sql_path)

Database: E:\ADY291c-AI2107-Group2\data\olist.db
SQL source: E:\ADY291c-AI2107-Group2\sql\03_queries.sql


In [21]:
# Tạo các view cần thiết; không chạy các câu truy vấn Q1-Q8 trong file SQL.
# Mỗi câu truy vấn phân tích được thực thi riêng ở cell bên dưới.
sql_text = sql_path.read_text(encoding="utf-8")
q1_marker = "-- Q1:"
leakage_marker = "-- LEAKAGE-SAFE MODEL FEATURE VIEWS"

view_setup = sql_text.split(q1_marker, 1)[0]
if leakage_marker in sql_text:
    view_setup += "\n" + leakage_marker + "\n"
    view_setup += sql_text.split(leakage_marker, 1)[1]

conn = sqlite3.connect(db_path, timeout=30)
conn.execute("PRAGMA cache_size = -64000")
conn.execute("PRAGMA temp_store = MEMORY")
conn.executescript(view_setup)

views = conn.execute(
    "SELECT name FROM sqlite_master WHERE type='view' ORDER BY name"
).fetchall()

print("Danh sách view:")
for view_name, in views:
    print("-", view_name)

print("Số đơn phân tích:", conn.execute(
    "SELECT COUNT(*) FROM v_review_order_features"
).fetchone()[0])


Danh sách view:
- v_category_performance
- v_customer_rfm
- v_delivery_kpi
- v_model_features_at_purchase
- v_model_features_post_delivery
- v_payment_mix
- v_review_order_features
- v_sales_monthly
- v_seller_performance
Số đơn phân tích: 95329


## Quy ước dữ liệu

View `v_review_order_features` chứa một dòng cho mỗi đơn đã giao và đã có đánh giá. Dữ liệu sản phẩm và thanh toán được tổng hợp theo `order_id` trước khi JOIN để tránh nhân đôi đánh giá.

- `delay_days <= 0`: đơn giao đúng hạn hoặc sớm hơn dự kiến.
- `delay_days > 0`: đơn giao trễ so với ngày dự kiến.
- `review_score`: điểm đánh giá từ 1 đến 5.


## Q1: Phan bo va mat can bang review_score

**Mục tiêu:** Xem số lượng và tỷ lệ đơn theo từng mức đánh giá từ 1 đến 5.

**Cách thực hiện:** Kết quả cho biết nhóm điểm chiếm ưu thế và mức mất cân bằng trong dữ liệu đánh giá.

**Kết quả mong đợi:** DataFrame `df_q1` chứa dữ liệu đã tổng hợp.

In [24]:
q1 = """
SELECT review_score,
       COUNT(*) AS order_count,
       ROUND(100.0 * COUNT(*) /
             (SELECT COUNT(*) FROM v_review_order_features), 2) AS percentage
FROM v_review_order_features
GROUP BY review_score
ORDER BY review_score;
"""

df_q1 = pd.read_sql_query(q1, conn)
df_q1


,review_score,order_count,percentage
0,1,9277,9.73
1,2,2900,3.04
2,3,7865,8.25
3,4,18807,19.73
4,5,56480,59.25


### Kết luận Q1 và bước tiếp theo

- **Kết luận và bước tiếp theo:** Q1 cho thấy phân bố review_score có thể mất cân bằng mạnh; kết luận mô tả chưa đủ để coi các mức khác nhau là có ý nghĩa kinh doanh. Bước tiếp theo: vẽ count plot/bar chart tỷ lệ từng điểm, dùng stratified split và metric như macro-F1 khi mô hình hóa.


## Q2: Don giao tre so voi dung han

**Mục tiêu:** So sánh tỷ lệ đánh giá tốt và không tốt giữa đơn giao trễ và đơn giao đúng hạn.

**Cách thực hiện:** Nhóm giao trễ được xác định bằng `delay_days > 0`.

**Kết quả mong đợi:** DataFrame `df_q2` chứa dữ liệu đã tổng hợp.

In [25]:
q2 = """
SELECT CASE WHEN is_late = 1 THEN 'Giao tre'
            ELSE 'Dung han hoac som' END AS delivery_group,
       COUNT(*) AS order_count,
       ROUND(AVG(review_score), 2) AS avg_review_score,
       ROUND(100.0 * AVG(CASE WHEN review_score >= 4 THEN 1 ELSE 0 END), 2) AS positive_pct,
       ROUND(100.0 * AVG(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END), 2) AS low_score_pct
FROM v_review_order_features
GROUP BY is_late
ORDER BY delivery_group;
"""

df_q2 = pd.read_sql_query(q2, conn)
df_q2


,delivery_group,order_count,avg_review_score,positive_pct,low_score_pct
0,Dung han hoac som,88985,4.29,82.71,9.23
1,Giao tre,6344,2.27,26.66,62.42


### Kết luận Q2 và bước tiếp theo

- **Kết luận và bước tiếp theo:** Q2 cho thấy trạng thái giao đúng hạn/sớm và giao trễ có khả năng gắn với mức review khác nhau rõ rệt. Bước tiếp theo: vẽ boxplot/violin và stacked bar is_late × review_score; biến này là đặc trưng hậu giao hàng, không dùng cho mô hình đặt hàng.


## Q3: Muc do tre theo ngay

**Mục tiêu:** Phân tích mức độ giao trễ theo từng mốc thời gian đặt hàng.

**Cách thực hiện:** Câu truy vấn gom đơn theo ngày hoặc chu kỳ tương ứng để quan sát xu hướng trễ.

**Kết quả mong đợi:** DataFrame `df_q3` chứa dữ liệu đã tổng hợp.

In [26]:
q3 = """
SELECT CASE
           WHEN delay_days <= 0 THEN 'Som hoac dung han'
           WHEN delay_days <= 3 THEN 'Tre 1-3 ngay'
           WHEN delay_days <= 7 THEN 'Tre 4-7 ngay'
           ELSE 'Tre tren 7 ngay'
       END AS delay_group,
       COUNT(*) AS order_count,
       ROUND(AVG(review_score), 2) AS avg_review_score,
       ROUND(100.0 * AVG(CASE WHEN review_score >= 4 THEN 1 ELSE 0 END), 2) AS positive_pct
FROM v_review_order_features
WHERE delay_days IS NOT NULL
GROUP BY delay_group
ORDER BY MIN(CASE
                 WHEN delay_days <= 0 THEN 0
                 WHEN delay_days <= 3 THEN 1
                 WHEN delay_days <= 7 THEN 2
                 ELSE 3 END);
"""

df_q3 = pd.read_sql_query(q3, conn)
df_q3


,delay_group,order_count,avg_review_score,positive_pct
0,Som hoac dung han,88985,4.29,82.71
1,Tre 1-3 ngay,1841,3.29,53.18
2,Tre 4-7 ngay,1740,2.10,22.41
3,Tre tren 7 ngay,2763,1.69,11.65


### Kết luận Q3 và bước tiếp theo

- **Kết luận và bước tiếp theo:** Q3 phân nhóm mức độ trễ giúp quan sát xem review giảm thế nào khi thời gian trễ tăng. Bước tiếp theo: vẽ line/bar chart theo bucket trễ và kiểm tra ngưỡng ảnh hưởng; chuyển bucket thành ordered category hoặc độ trễ liên tục khi cần.


## Q4: Phi ship va ty le phi/gia

**Mục tiêu:** Đánh giá mối liên hệ giữa phí vận chuyển, tỷ lệ phí/gía trị hàng và mức đánh giá.

**Cách thực hiện:** Dữ liệu được tổng hợp theo nhóm phân tích để kiểm tra chi phí và mức hài lòng.

**Kết quả mong đợi:** DataFrame `df_q4` chứa dữ liệu đã tổng hợp.

In [27]:
q4 = """
-- Q4: Phân tích hai chỉ số phí ship đối với review_score.
WITH ship_fee_data AS (
    -- Mỗi dòng là một đơn đã giao và đã có review.
    SELECT total_freight AS metric_value, review_score
    FROM v_review_order_features
),
ship_fee_quartiles AS (
    -- NTILE(4) chia đơn thành 4 nhóm gần bằng nhau theo tổng phí ship.
    SELECT metric_value, review_score,
           NTILE(4) OVER (ORDER BY metric_value) AS quartile_number
    FROM ship_fee_data
),
ship_fee_ratio_data AS (
    SELECT freight_to_price_pct AS metric_value, review_score
    FROM v_review_order_features
    WHERE freight_to_price_pct IS NOT NULL
),
ship_fee_ratio_quartiles AS (
    -- NTILE(4) chia đơn theo tỷ lệ phí ship/gía trị hàng hóa.
    SELECT metric_value, review_score,
           NTILE(4) OVER (ORDER BY metric_value) AS quartile_number
    FROM ship_fee_ratio_data
),
ship_fee_summary AS (
    SELECT
        'Tong phi ship (BRL)' AS chi_so,
        CASE quartile_number
            WHEN 1 THEN '1 - 25% phi thap nhat'
            WHEN 2 THEN '2 - Tu 25%-50%'
            WHEN 3 THEN '3 - Tu 50%-75%'
            ELSE '4 - 25% phi cao nhat'
        END AS nhom_phi,
        quartile_number AS nhom_quartile,
        COUNT(*) AS so_don,
        ROUND(AVG(metric_value), 2) AS gia_tri_trung_binh,
        ROUND(AVG(review_score), 2) AS diem_review_tb,
        ROUND(100.0 * AVG(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END), 2)
            AS ty_le_review_1_2_pct
    FROM ship_fee_quartiles
    GROUP BY quartile_number
),
ship_fee_ratio_summary AS (
    SELECT
        'Ty le phi ship/gia hang (%)' AS chi_so,
        CASE quartile_number
            WHEN 1 THEN '1 - 25% ty le thap nhat'
            WHEN 2 THEN '2 - Tu 25%-50%'
            WHEN 3 THEN '3 - Tu 50%-75%'
            ELSE '4 - 25% ty le cao nhat'
        END AS nhom_phi,
        quartile_number AS nhom_quartile,
        COUNT(*) AS so_don,
        ROUND(AVG(metric_value), 2) AS gia_tri_trung_binh,
        ROUND(AVG(review_score), 2) AS diem_review_tb,
        ROUND(100.0 * AVG(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END), 2)
            AS ty_le_review_1_2_pct
    FROM ship_fee_ratio_quartiles
    GROUP BY quartile_number
)
-- Ghep hai bang tom tat theo tung metric, giu thu tu quartile 1 den 4.
SELECT * FROM ship_fee_summary
UNION ALL
SELECT * FROM ship_fee_ratio_summary
ORDER BY chi_so DESC, nhom_quartile;
"""

df_q4 = pd.read_sql_query(q4, conn)
df_q4


,chi_so,nhom_phi,nhom_quartile,so_don,gia_tri_trung_binh,diem_review_tb,ty_le_review_1_2_pct
0,Ty le phi ship/gia hang (%),1 - 25% ty le thap nhat,1,23833,8.41,4.20,12.32
1,Ty le phi ship/gia hang (%),2 - Tu 25%-50%,2,23832,17.55,4.17,12.35
2,Ty le phi ship/gia hang (%),3 - Tu 50%-75%,3,23832,29.18,4.15,13.02
3,Ty le phi ship/gia hang (%),4 - 25% ty le cao nhat,4,23832,68.17,4.12,13.40
4,Tong phi ship (BRL),1 - 25% phi thap nhat,1,23833,10.44,4.32,9.09
5,Tong phi ship (BRL),2 - Tu 25%-50%,2,23832,15.46,4.19,11.77
6,Tong phi ship (BRL),3 - Tu 50%-75%,3,23832,19.77,4.18,12.09
7,Tong phi ship (BRL),4 - 25% phi cao nhat,4,23832,45.39,3.94,18.15


### Kết luận Q4 và bước tiếp theo

- **Kết luận và bước tiếp theo:** Q4 dùng NTILE(4) chia đơn thành 4 nhóm phí ship gần bằng nhau để so sánh. Bước tiếp theo: vẽ bar chart review theo nhóm phí ship và scatter phí ship/tỷ lệ phí ship với review; đồng thời kiểm tra tương quan với giá trị đơn và khoảng cách địa lý.


## Q5: Do phuc tap don hang

**Mục tiêu:** So sánh mức hài lòng theo số sản phẩm, số loại sản phẩm và số người bán trong đơn.

**Cách thực hiện:** Các đặc trưng của đơn được tổng hợp trước khi phân nhóm để tránh nhân đôi dữ liệu.

**Kết quả mong đợi:** DataFrame `df_q5` chứa dữ liệu đã tổng hợp.

In [28]:
q5 = """
SELECT CASE
           WHEN item_count = 1 THEN '1 mat hang'
           WHEN item_count BETWEEN 2 AND 3 THEN '2-3 mat hang'
           ELSE '4+ mat hang'
       END AS basket_group,
       CASE WHEN seller_count = 1 THEN '1 nguoi ban'
            ELSE 'Nhieu nguoi ban' END AS seller_group,
       COUNT(*) AS order_count,
       ROUND(AVG(review_score), 2) AS avg_review_score,
       ROUND(100.0 * AVG(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END), 2) AS low_score_pct
FROM v_review_order_features
GROUP BY basket_group, seller_group
ORDER BY MIN(item_count), MIN(seller_count);
"""

df_q5 = pd.read_sql_query(q5, conn)
df_q5


,basket_group,seller_group,order_count,avg_review_score,low_score_pct
0,1 mat hang,1 nguoi ban,85871,4.21,11.24
1,2-3 mat hang,1 nguoi ban,7408,3.78,22.80
2,2-3 mat hang,Nhieu nguoi ban,1132,2.89,46.11
3,4+ mat hang,1 nguoi ban,804,3.48,30.60
4,4+ mat hang,Nhieu nguoi ban,114,2.49,59.65


### Kết luận Q5 và bước tiếp theo

- **Kết luận và bước tiếp theo:** Q5 cho phép xem mức hài lòng thay đổi theo độ phức tạp đơn: số mặt hàng, số sản phẩm và số người bán. Bước tiếp theo: vẽ heatmap/bar chart độ phức tạp, tính low-score rate và tạo các tương tác feature cho phân tích Python.


## Q6: Thanh toan va tra gop

**Mục tiêu:** Phân tích ảnh hưởng của hình thức thanh toán và số kỳ trả góp đến điểm đánh giá.

**Cách thực hiện:** Kết quả đối chiếu số đơn và điểm đánh giá trung bình trong từng nhóm thanh toán.

**Kết quả mong đợi:** DataFrame `df_q6` chứa dữ liệu đã tổng hợp.

In [29]:
q6 = """
SELECT first_payment_type AS payment_type,
       COUNT(*) AS order_count,
       ROUND(AVG(max_installments), 2) AS avg_installments,
       ROUND(AVG(review_score), 2) AS avg_review_score,
       ROUND(100.0 * AVG(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END), 2) AS low_score_pct
FROM v_review_order_features
WHERE first_payment_type IS NOT NULL
GROUP BY first_payment_type
ORDER BY order_count DESC;
"""

df_q6 = pd.read_sql_query(q6, conn)
df_q6


,payment_type,order_count,avg_installments,avg_review_score,low_score_pct
0,credit_card,73387,3.5,4.16,12.84
1,boleto,18970,1.0,4.16,12.51
2,voucher,1471,1.0,4.11,14.28
3,debit_card,1424,1.0,4.25,10.88


### Kết luận Q6 và bước tiếp theo

- **Kết luận và bước tiếp theo:** Q6 tổng hợp hành vi thanh toán và số kỳ trả góp theo từng đơn. Bước tiếp theo: vẽ payment mix, stacked bar review theo phương thức và phân phối installment; đối chiếu giá trị/ phí ship để tránh kết luận sai do quy mô nhóm.


## Q7: Xu huong review theo thang

**Mục tiêu:** Quan sát biến động điểm đánh giá trung bình và số lượng review theo từng tháng.

**Cách thực hiện:** Kết quả được sắp xếp theo năm-tháng để nhận diện xu hướng thời gian.

**Kết quả mong đợi:** DataFrame `df_q7` chứa dữ liệu đã tổng hợp.

In [30]:
q7 = """
SELECT purchase_year_month,
       COUNT(*) AS order_count,
       ROUND(AVG(review_score), 2) AS avg_review_score,
       ROUND(100.0 * AVG(CASE WHEN review_score >= 4 THEN 1 ELSE 0 END), 2) AS positive_pct,
       ROUND(100.0 * AVG(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END), 2) AS low_score_pct
FROM v_review_order_features
GROUP BY purchase_year_month
ORDER BY purchase_year_month;
"""

df_q7 = pd.read_sql_query(q7, conn)
df_q7


,purchase_year_month,order_count,avg_review_score,positive_pct,low_score_pct
0,2016-10,260,4.01,75.38,16.54
1,2016-12,1,5.00,100.00,0.00
2,2017-01,729,4.20,79.84,11.93
3,2017-02,1634,4.21,79.44,11.02
4,2017-03,2506,4.19,79.93,11.13
5,2017-04,2276,4.13,78.08,13.01
6,2017-05,3490,4.24,80.97,10.00
7,2017-06,3083,4.22,80.60,10.70
8,2017-07,3809,4.26,81.46,9.90
9,2017-08,4127,4.32,83.81,9.33


### Kết luận Q7 và bước tiếp theo

- **Kết luận và bước tiếp theo:** Q7 mô tả xu hướng review và số đơn theo tháng trong 22 tháng. Bước tiếp theo: vẽ line chart, rolling average và biểu đồ review tốt/thấp; kiểm tra mùa vụ và sự kiện có thể giải thích biến động.


## Q8: Hai long theo bang/khu vuc

**Mục tiêu:** So sánh mức hài lòng theo bang hoặc khu vực của khách hàng.

**Cách thực hiện:** Kết quả gom dữ liệu theo địa lý để tìm khu vực có đánh giá cao hoặc thấp.

**Kết quả mong đợi:** DataFrame `df_q8` chứa dữ liệu đã tổng hợp.

In [31]:
q8 = """
SELECT customer_state,
       COUNT(*) AS order_count,
       ROUND(AVG(review_score), 2) AS avg_review_score,
       ROUND(100.0 * AVG(CASE WHEN review_score >= 4 THEN 1 ELSE 0 END), 2) AS positive_pct,
       ROUND(100.0 * AVG(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END), 2) AS low_score_pct,
       ROUND(AVG(is_late) * 100.0, 2) AS late_rate_pct,
       ROUND(AVG(delivery_days), 2) AS avg_delivery_days
FROM v_review_order_features
GROUP BY customer_state
HAVING COUNT(*) >= 100
ORDER BY avg_review_score ASC, low_score_pct DESC;
"""

df_q8 = pd.read_sql_query(q8, conn)
df_q8


,customer_state,order_count,avg_review_score,positive_pct,low_score_pct,late_rate_pct,avg_delivery_days
0,MA,706,3.82,69.97,20.11,17.28,21.44
1,AL,393,3.85,71.76,21.12,20.61,24.41
2,SE,332,3.90,72.29,18.98,15.06,21.46
3,PA,924,3.92,72.40,17.86,10.82,23.64
4,BA,3213,3.93,72.92,16.99,11.86,19.23
5,CE,1272,3.94,72.80,17.14,13.68,21.22
6,RJ,12136,3.97,73.48,18.27,11.93,15.24
7,PI,470,3.99,75.11,16.17,13.62,19.47
8,PB,510,4.07,75.88,14.90,10.20,20.26
9,PE,1573,4.08,77.24,14.88,9.47,18.38


### Kết luận Q8 và bước tiếp theo

- **Kết luận và bước tiếp theo:** Q8 cho biết mức hài lòng, tỷ lệ giao trễ và thời gian giao phân hóa theo bang khách hàng. Bước tiếp theo: vẽ choropleth bản đồ Brazil, ranking bang và scatter late rate/review để ưu tiên tuyến logistics cần cải thiện.


# Bảng tổng hợp cho bước khám phá bằng Python

Bảng dưới đây chuyển kết quả SQL thành đầu việc trực quan hóa và khám phá hành vi mua sắm bằng Python: bản đồ khách hàng, phương thức thanh toán và tương quan.

| Truy vấn | Output SQL đã có | Trực quan Python phù hợp | Giá trị cho bước tiếp theo |
|---|---|---|---|
| Q1 | Số lượng và tỷ lệ từng mức review | Count plot/bar chart tỷ lệ 1-5 | Chọn biểu đồ, stratified split và metric khi mô hình hóa |
| Q2 | Review theo đúng hạn/sớm và trễ | Violin/boxplot và stacked bar | Kiểm tra tác động giao trễ; biến hậu giao hàng |
| Q3 | Review theo bucket mức độ trễ | Line/bar chart theo bucket | Xác định ngưỡng trễ ảnh hưởng mạnh nhất |
| Q4 | Quartile phí ship và tỷ lệ phí ship | Bar chart theo quartile, scatter và correlation | Kiểm tra chi phí ship liên quan đến review và giá trị đơn |
| Q5 | Review theo số item, sản phẩm và seller | Heatmap/bar chart độ phức tạp | Phân tích đơn phức tạp và tạo feature tương tác |
| Q6 | Payment type, installment và review | Payment mix, stacked bar và histogram | Trực quan hóa hành vi thanh toán theo nhóm khách hàng |
| Q7 | Review và số đơn theo tháng | Line chart 22 tháng và rolling average | Phát hiện xu hướng, mùa vụ và thời điểm mua sắm |
| Q8 | Review, trễ và thời gian giao theo bang | Choropleth Brazil, ranking và scatter | Xác định vùng logistics cần cải thiện trên bản đồ |

## Hướng dùng cho EDA Python
- Ưu tiên trực quan các yếu tố có khác biệt lớn: giao trễ, mức độ trễ, độ phức tạp và khu vực.
- Dùng bản đồ bang/khu vực kết hợp tỷ lệ giao trễ và điểm review thấp để tìm vị trí cần tối ưu logistics.
- Dùng phương thức thanh toán và số kỳ trả góp để phân tích hành vi thanh toán theo nhóm khách hàng.
- Dùng heatmap tương quan và scatter plot cho phí ship, giá trị đơn, khoảng cách, thời gian giao và review.
- Không dùng đặc trưng sau giao hàng như delay_days, is_late, delivery_days cho mô hình dự đoán tại thời điểm đặt hàng.


In [32]:
# Đóng kết nối sau khi hoàn tất toàn bộ truy vấn.
conn.close()
